# Neural Network Modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score

import sys
from pathlib import Path

In [ ]:
# Add the project root directory to the Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.feature.group_rare_cat import RareCategoryGrouper

In [3]:
train_df = pd.read_csv('../data/processed/train_data_processed.csv')
test_value = pd.read_csv('../data/processed/test_values_processed.csv')

train_df.head()

,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,...,has_superstructure_rc_engineered,has_superstructure_other,legal_ownership_status,count_families,has_secondary_use_other,damage_grade,height_area_ratio,has_secondary_use_manufacture,has_secondary_use_public,has_secondary_use_business
0,6,487,12198,2,30.0,t,r,n,f,q,...,0,0,v,1,0,3,0.833333,0,0,0
1,8,900,2812,2,10.0,o,r,n,x,q,...,0,0,v,1,0,2,0.875000,0,0,0
2,21,363,8973,2,10.0,t,r,n,f,x,...,0,0,v,1,0,3,1.000000,0,0,0
3,22,418,10694,2,10.0,t,r,n,f,x,...,0,0,v,1,0,2,0.833333,0,0,0
4,11,131,1488,3,30.0,t,r,n,f,x,...,0,0,v,1,0,3,1.125000,0,0,0


### Train-Test Split

In [ ]:
# Separate data and target
y = train_df['damage_grade']
X = train_df.drop(['damage_grade'], axis=1)

# Split the data
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)  # Keep the class distribution of the target variable consistent

X_train

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other
185461,716310,21,935,5965,3,20,6,6,t,r,...,0,0,0,0,0,0,0,0,0,0
190537,283080,3,1229,1099,2,20,6,5,t,r,...,0,0,0,0,0,0,0,0,0,0
232781,690774,21,1219,2705,3,45,6,5,t,r,...,0,0,0,0,0,0,0,0,0,0
213004,904803,20,158,2452,2,10,7,5,t,w,...,0,0,0,0,0,0,0,0,0,0
177786,991971,9,670,5977,2,0,9,6,t,i,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99791,819626,8,1184,830,1,10,6,5,t,r,...,0,0,0,0,0,0,0,0,0,0
150155,865223,4,144,5751,2,0,8,4,t,r,...,0,0,0,0,0,0,0,0,0,0
112797,363536,27,548,1119,3,20,19,7,t,r,...,0,0,0,0,0,0,0,0,0,0
225588,402196,17,1030,606,2,15,8,8,t,r,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
X_test = test_value.copy()
X_test

,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,...,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,legal_ownership_status,count_families,has_secondary_use_other,has_secondary_use_manufacture,has_secondary_use_public,has_secondary_use_business,height_area_ratio
0,17,596,11307,3,20.0,t,r,n,f,q,...,0,0,0,v,1,0,0,0,0,0.857143
1,6,141,11987,2,25.0,t,r,n,f,q,...,0,0,0,v,1,0,1,0,0,0.384615
2,22,19,10044,2,5.0,t,r,n,f,q,...,0,0,0,v,1,0,0,0,0,1.250000
3,26,39,633,1,0.0,t,r,x,v,j,...,0,0,0,v,2,0,0,0,1,0.157895
4,17,289,7970,3,15.0,t,r,q,f,q,...,0,0,0,v,1,0,0,0,0,0.875000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86863,4,605,3623,3,70.0,t,r,q,f,q,...,0,0,0,other,1,0,1,0,0,0.300000
86864,10,1407,11907,3,25.0,n,r,n,f,q,...,0,0,0,v,1,0,0,0,0,1.166667
86865,22,1136,7712,1,50.0,t,r,n,f,j,...,0,0,0,v,1,0,0,0,0,1.000000
86866,6,1041,912,2,5.0,t,r,n,f,q,...,0,0,0,a,1,0,0,0,0,0.555556


In [ ]:
# Binary columns

binary_cols = ['has_superstructure_adobe_mud',
       'has_superstructure_mud_mortar_stone', 'has_superstructure_stone_flag',
       'has_superstructure_cement_mortar_stone',
       'has_superstructure_mud_mortar_brick',
       'has_superstructure_cement_mortar_brick', 'has_superstructure_timber',
       'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered',
       'has_superstructure_rc_engineered', 'has_superstructure_other',
        'has_secondary_use_other',
       'has_secondary_use_manufacture',
       'has_secondary_use_public', 'has_secondary_use_business']

# Select binary features
X_train_binary = X_train[binary_cols].copy()
X_validation_binary = X_validation[binary_cols].copy()
X_test_binary = X_test[binary_cols].copy()

In [ ]:
# Define a function to convert data to torch tensors
def to_tensor(data):
    return torch.as_tensor(data, dtype=torch.float32)

In [ ]:
# Convert binary features to torch tensors
X_train_binary = to_tensor(X_train_binary.values)

X_validation_binary = to_tensor(X_validation_binary.values)

X_test_binary = to_tensor(X_test_binary.values)

In [ ]:
# Numerical columns

num_cols = ['count_floors_pre_eq', 'age', 'count_families', 'height_area_ratio']

# Select numerical features
X_train_num = X_train[num_cols].copy()
X_validation_num = X_validation[num_cols].copy()
X_test_num = X_test[num_cols].copy()

# Build numerical features preprocessing pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler())
])

# Fit on training data only
X_train_num = num_pipeline.fit_transform(X_train_num)

# Apply the same transformations to validation and test data
X_validation_num = num_pipeline.transform(X_validation_num)
X_test_num = num_pipeline.transform(X_test_num)

# Convert numerical features to torch tensors
X_train_num = to_tensor(X_train_num)
X_validation_num = to_tensor(X_validation_num)
X_test_num = to_tensor(X_test_num)

# Combine numerical and binary features
X_train_numerical = torch.cat([X_train_num, X_train_binary], dim=1)
X_validation_numerical = torch.cat([X_validation_num, X_validation_binary], dim=1)
X_test_numerical = torch.cat([X_test_num, X_test_binary], dim=1)

In [ ]:
# Categorical columns

cat_cols = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id',
       'land_surface_condition',
       'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type',
       'position', 'plan_configuration', 
       'legal_ownership_status']

# Select categorical features
X_train_cat = X_train[cat_cols].copy()
X_validation_cat = X_validation[cat_cols].copy()
X_test_cat = X_test[cat_cols].copy()

In [ ]:
# Rare Category Grouper
grouper = RareCategoryGrouper()

# Fit the grouper on training data and transform all datasets
X_train_cat = grouper.fit_transform(X_train_cat)
X_validation_cat = grouper.transform(X_validation_cat)
X_test_cat = grouper.transform(X_test_cat)

In [ ]:
# Integer Encoding
from sklearn.preprocessing import OrdinalEncoder

# Define the encoder with handling for unknown categories
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",  # unknown categories
    unknown_value=-1
)

# Fit the encoder on training data and transform all datasets
X_train_cat = encoder.fit_transform(X_train_cat)
X_validation_cat = encoder.transform(X_validation_cat)
X_test_cat = encoder.transform(X_test_cat)

In [ ]:
# nn.Embedding cannot directly accept -1, so we increase all values by 1

X_train_cat = X_train_cat + 1
X_validation_cat = X_validation_cat + 1
X_test_cat = X_test_cat + 1

In [ ]:
# Convert categorical features to torch tensors
X_train_cat = torch.tensor(
    X_train_cat,
    dtype=torch.long
)

X_validation_cat = torch.tensor(
    X_validation_cat,
    dtype=torch.long
)

X_test_cat = torch.tensor(
    X_test_cat,
    dtype=torch.long
)

In [ ]:
# Check the shapes of the processed datasets
print(X_train_num.shape)
print(X_train_binary.shape)
print(X_train_numerical.shape)
print(X_train_cat.shape)

torch.Size([208480, 4])
torch.Size([208480, 15])
torch.Size([208480, 19])
torch.Size([208480, 11])


In [ ]:
for i, col in enumerate(cat_cols):
    print(col, len(encoder.categories_[i]))

geo_level_1_id 25
geo_level_2_id 1409
geo_level_3_id 11380
land_surface_condition 3
foundation_type 5
roof_type 3
ground_floor_type 4
other_floor_type 4
position 4
plan_configuration 4
legal_ownership_status 3


In [ ]:
# Define embedding sizes for each categorical feature
embedding_sizes = [
    (26, 8),
    (1410, 32),
    (11381, 32),
    (4, 4),
    (6, 4),
    (4, 4),
    (5, 4),
    (5, 4),
    (5, 4),
    (5, 4),
    (4, 4)
]

In [ ]:
# Convert target variable to torch tensors
y_train_tensor = torch.tensor(
    y_train.values - 1, # Convert to 0-based indexing
    dtype=torch.long
)

y_validation_tensor = torch.tensor(
    y_validation.values - 1,
    dtype=torch.long
)

In [ ]:
# Define a custom dataset class for PyTorch
class EarthquakeDataset(Dataset):

    def __init__(self, numerical, categorical, target):
        self.numerical = numerical
        self.categorical = categorical
        self.target = target

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):
        return (
            self.numerical[idx],
            self.categorical[idx],
            self.target[idx]
        )

In [ ]:
# Create DataLoader instances for training and validation datasets

train_dataset = EarthquakeDataset(
    X_train_numerical,
    X_train_cat,
    y_train_tensor
)

validation_dataset = EarthquakeDataset(
    X_validation_numerical,
    X_validation_cat,
    y_validation_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=512,
    shuffle=False
)

### Model Training

In [4]:
# Define the neural network model
class EarthquakeModel(nn.Module):

    def __init__(self, embedding_sizes, numerical_input_size):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, embedding_dim)
            for num_categories, embedding_dim in embedding_sizes
        ])

        embedding_output_size = sum(
            embedding_dim
            for _, embedding_dim in embedding_sizes
        )

        self.mlp = nn.Sequential(
            nn.Linear(numerical_input_size + embedding_output_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, numerical, categorical):

        embedded = [
            embedding(categorical[:, i])
            for i, embedding in enumerate(self.embeddings)
        ]

        x_cat = torch.cat(embedded, dim=1)

        x = torch.cat([numerical, x_cat], dim=1)

        x = self.mlp(x)

        return x

In [ ]:
model = EarthquakeModel(
    embedding_sizes=embedding_sizes,
    numerical_input_size=19
)

In [ ]:
# Check the shapes of the inputs and outputs
for numerical, categorical, target in train_loader:

    print("numerical:", numerical.shape)
    print("categorical:", categorical.shape)
    print("target:", target.shape)

    output = model(numerical, categorical)

    print("output:", output.shape)

    break

numerical: torch.Size([512, 19])
categorical: torch.Size([512, 11])
target: torch.Size([512])
output: torch.Size([512, 3])


In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

In [ ]:
# Check the loss computation
for numerical, categorical, target in train_loader:

    output = model(numerical, categorical)

    loss = criterion(output, target)

    print("output:", output.shape)
    print("target:", target.shape)
    print("loss:", loss.item())

    break

output: torch.Size([512, 3])
target: torch.Size([512])
loss: 1.108485460281372


In [ ]:
# Define the optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
# Set training parameters
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Set the number of epochs and early stopping patience
num_epochs = 30
patience = 5

# Initialize variables for early stopping
best_f1 = 0
epochs_without_improvement = 0

# Training loop
for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================

    model.train()

    total_loss = 0

    for numerical, categorical, target in train_loader:

        optimizer.zero_grad()

        output = model(numerical, categorical)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)


    # =====================
    # Validation
    # =====================

    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for numerical, categorical, target in validation_loader:

            output = model(numerical, categorical)

            predictions = output.argmax(dim=1)

            all_predictions.append(predictions)
            all_targets.append(target)


    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)


    validation_f1 = f1_score(
        all_targets.numpy(),
        all_predictions.numpy(),
        average="micro"
    )


    # =====================
    # Print results
    # =====================

    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {average_loss:.4f} | "
        f"Validation Micro-F1: {validation_f1:.4f}"
    )


    # =====================
    # Save best model
    # =====================

    if validation_f1 > best_f1:

        best_f1 = validation_f1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

        print("  → Best model saved!")

    else:

        epochs_without_improvement += 1

        print(
            f"  → No improvement "
            f"({epochs_without_improvement}/{patience})"
        )


    # =====================
    # Early stopping
    # =====================

    if epochs_without_improvement >= patience:

        print("Early stopping!")

        break

Epoch 1 | Train Loss: 0.6955 | Validation Micro-F1: 0.7216
  → Best model saved!
Epoch 2 | Train Loss: 0.6062 | Validation Micro-F1: 0.7326
  → Best model saved!
Epoch 3 | Train Loss: 0.5788 | Validation Micro-F1: 0.7381
  → Best model saved!
Epoch 4 | Train Loss: 0.5610 | Validation Micro-F1: 0.7407
  → Best model saved!
Epoch 5 | Train Loss: 0.5490 | Validation Micro-F1: 0.7453
  → Best model saved!
Epoch 6 | Train Loss: 0.5380 | Validation Micro-F1: 0.7412
  → No improvement (1/5)
Epoch 7 | Train Loss: 0.5296 | Validation Micro-F1: 0.7430
  → No improvement (2/5)
Epoch 8 | Train Loss: 0.5219 | Validation Micro-F1: 0.7440
  → No improvement (3/5)
Epoch 9 | Train Loss: 0.5147 | Validation Micro-F1: 0.7449
  → No improvement (4/5)
Epoch 10 | Train Loss: 0.5087 | Validation Micro-F1: 0.7439
  → No improvement (5/5)
Early stopping!


In [ ]:
# Adjust learning rate and number of epochs for further training

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

num_epochs = 50
patience = 15

best_f1 = 0
epochs_without_improvement = 0


for epoch in range(num_epochs):

    # =====================
    # Training
    # =====================

    model.train()

    total_loss = 0

    for numerical, categorical, target in train_loader:

        optimizer.zero_grad()

        output = model(numerical, categorical)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)


    # =====================
    # Validation
    # =====================

    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for numerical, categorical, target in validation_loader:

            output = model(numerical, categorical)

            predictions = output.argmax(dim=1)

            all_predictions.append(predictions)
            all_targets.append(target)


    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)


    validation_f1 = f1_score(
        all_targets.numpy(),
        all_predictions.numpy(),
        average="micro"
    )


    # =====================
    # Print results
    # =====================

    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {average_loss:.4f} | "
        f"Validation Micro-F1: {validation_f1:.4f}"
    )


    # =====================
    # Save best model
    # =====================

    if validation_f1 > best_f1:

        best_f1 = validation_f1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

        print("  → Best model saved!")

    else:

        epochs_without_improvement += 1

        print(
            f"  → No improvement "
            f"({epochs_without_improvement}/{patience})"
        )


    # =====================
    # Early stopping
    # =====================

    if epochs_without_improvement >= patience:

        print("Early stopping!")

        break

Epoch 1 | Train Loss: 0.7355 | Validation Micro-F1: 0.7107
  → Best model saved!
Epoch 2 | Train Loss: 0.6335 | Validation Micro-F1: 0.7276
  → Best model saved!
Epoch 3 | Train Loss: 0.6019 | Validation Micro-F1: 0.7346
  → Best model saved!
Epoch 4 | Train Loss: 0.5826 | Validation Micro-F1: 0.7379
  → Best model saved!
Epoch 5 | Train Loss: 0.5694 | Validation Micro-F1: 0.7422
  → Best model saved!
Epoch 6 | Train Loss: 0.5585 | Validation Micro-F1: 0.7428
  → Best model saved!
Epoch 7 | Train Loss: 0.5504 | Validation Micro-F1: 0.7436
  → Best model saved!
Epoch 8 | Train Loss: 0.5424 | Validation Micro-F1: 0.7431
  → No improvement (1/15)
Epoch 9 | Train Loss: 0.5362 | Validation Micro-F1: 0.7453
  → Best model saved!
Epoch 10 | Train Loss: 0.5304 | Validation Micro-F1: 0.7451
  → No improvement (1/15)
Epoch 11 | Train Loss: 0.5255 | Validation Micro-F1: 0.7445
  → No improvement (2/15)
Epoch 12 | Train Loss: 0.5207 | Validation Micro-F1: 0.7444
  → No improvement (3/15)
Epoch 13 

### Testset Prediction

In [ ]:
# Load the best model
model.load_state_dict(
    torch.load("../models/nn/best_model.pt")
)

model.eval()

EarthquakeModel(
  (embeddings): ModuleList(
    (0): Embedding(26, 8)
    (1): Embedding(1410, 32)
    (2): Embedding(11381, 32)
    (3): Embedding(4, 4)
    (4): Embedding(6, 4)
    (5): Embedding(4, 4)
    (6-9): 4 x Embedding(5, 4)
    (10): Embedding(4, 4)
  )
  (mlp): Sequential(
    (0): Linear(in_features=123, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)

In [ ]:
# Define a custom dataset class for PyTorch without target variable
class EarthquakeTestDataset(Dataset):

    def __init__(self, numerical, categorical):
        self.numerical = numerical
        self.categorical = categorical

    def __len__(self):
        return len(self.numerical)

    def __getitem__(self, idx):
        return (
            self.numerical[idx],
            self.categorical[idx]
        )

In [ ]:
# Create DataLoader instance for test dataset
test_dataset = EarthquakeTestDataset(
    X_test_numerical,
    X_test_cat
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)

In [ ]:
# Make predictions on the test dataset
all_predictions = []

model.eval()

with torch.no_grad():

    for numerical, categorical in test_loader:

        output = model(
            numerical,
            categorical
        )

        predictions = output.argmax(dim=1)

        all_predictions.append(predictions)


all_predictions = torch.cat(all_predictions)

In [ ]:
# Convert predictions to the original damage grade scale (1, 2, 3)
predicted_damage_grade = all_predictions.numpy() + 1

In [ ]:
# Save the submission file
submission_format = pd.read_csv('../data/submission_format.csv', index_col='building_id')
submission_format["damage_grade"] = predicted_damage_grade

print(submission_format.head())
print(submission_format["damage_grade"].value_counts())

             damage_grade
building_id              
300051                  3
99355                   2
890251                  2
745817                  1
421793                  3
damage_grade
2    55283
3    24258
1     7327
Name: count, dtype: int64


In [ ]:
# Save the submission file
submission_format.reset_index().to_csv("../submissions/submission_nn_01.csv", index=False)

Submission score: 0.7402

**Final Results**  
The fine-tuned Random Forest model with an ordinal classifier achieved the best micro-averaged F1 score of 0.7402, making an improvement of 0.1587 over the baseline.